In [1]:
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'datasets', 'lxml', 'cairosvg',
    'tokenizers', 'sentencepiece', 'tqdm', 'numpy'
], check=True)
print("Packages installed.")

Packages installed.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os, re, json
from lxml import etree
from datasets import load_dataset
from tqdm import tqdm
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import TemplateProcessing

BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'

DIRS = [
    'data/clean',
    'data/tokenized',
    'tokenizer',
    'checkpoints',
    'logs',
    'samples',
    'configs',
]

for d in DIRS:
    os.makedirs(f'{BASE_DIR}/{d}', exist_ok=True)

print(f"\nAll directories ready under: {BASE_DIR}")


All directories ready under: /content/drive/MyDrive/svg-lm-scaling


In [16]:
# Maximal Update Parameterization) with LR sweep + scaling train
# Reference: Yang et al. 2022
# Saves: /results/lr_sweep_mup.json + /results/mup_scaling_results.json

# Install mup if not already present
import subprocess
subprocess.run(['pip', 'install', '-q', 'mup'], check=True)
print("mup installed.")

import os, json, math, time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass
from mup import make_base_shapes, set_base_shapes, MuReadout
from mup.optim import MuAdamW

BASE_DIR    = '/content/drive/MyDrive/svg-lm-scaling'
DATA_DIR    = f'{BASE_DIR}/data/tokenized'
TOK_DIR     = f'{BASE_DIR}/tokenizer'
CKPT_DIR    = f'{BASE_DIR}/checkpoints'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(CKPT_DIR, exist_ok=True)

# Hyperparameters

BLOCK_SIZE    = 512
MICRO_BATCH   = 16
GRAD_ACCUM    = 8
SWEEP_STEPS   = 400
WARMUP_FRAC   = 0.05
LOG_INTERVAL  = 100
EVAL_INTERVAL = 200
CKPT_INTERVAL = 200   # incremental checkpoint every N steps (scaling runs only)
LR_CANDIDATES = [1e-4, 3e-4, 6e-4, 1e-3, 3e-3, 6e-3, 1e-2]

# mup base width must be strictly less than all target widths
MUP_BASE_WIDTH = 64

device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_bf16 = device == 'cuda' and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"Device: {device}  |  AMP: {amp_dtype}")

# Load config

with open(f'{TOK_DIR}/token_config.json') as f:
    tok_cfg = json.load(f)
VOCAB_SIZE = tok_cfg['vocab_size']

print("Loading data")
train_data = np.load(f'{DATA_DIR}/train.npy', mmap_mode='r')
val_data   = np.load(f'{DATA_DIR}/val.npy',   mmap_mode='r')
print(f"train: {len(train_data):,} tokens  |  val: {len(val_data):,} tokens")

STEPS_PER_EPOCH = len(train_data) // (MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE)

# mup-compatible GPT model
# Key differences from standard GPT:
#   1. Attention scale = 1/d_head  (not 1/sqrt(d_head))  mup rule for attention
#   2. lm_head = MuReadout (not nn.Linear) which is mup rule for output layer
# All other linear layers remain standard and set_base_shapes handles their LR scaling.

@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 4096
    n_layer:    int = 4
    n_head:     int = 4
    n_embd:     int = 128
    dropout:    float = 0.0
    bias:       bool = False

MODEL_CONFIGS = {
    'tiny':   GPTConfig(n_embd=128, n_layer=4,  n_head=4,  dropout=0.0),
    'small':  GPTConfig(n_embd=192, n_layer=6,  n_head=6,  dropout=0.0),
    'medium': GPTConfig(n_embd=384, n_layer=6,  n_head=6,  dropout=0.0),
    'large':  GPTConfig(n_embd=512, n_layer=10, n_head=8,  dropout=0.0),
    'xl':     GPTConfig(n_embd=768, n_layer=12, n_head=12, dropout=0.0),
}


class MuCausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn  = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_drop  = nn.Dropout(config.dropout)
        self.resid_drop = nn.Dropout(config.dropout)
        self.n_head  = config.n_head
        self.n_embd  = config.n_embd
        self.dropout = config.dropout
        self.flash   = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer(
                'causal_mask',
                torch.tril(torch.ones(config.block_size, config.block_size))
                     .view(1, 1, config.block_size, config.block_size)
            )

    def forward(self, x):
        B, T, C = x.size()
        hs = C // self.n_head
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        # mup attention scale: 1/d_head instead of 1/sqrt(d_head)
        mup_scale = 1.0 / hs
        if self.flash:
            # PyTorch SDPA uses sqrt(d) internally so we undo that and apply mup scale
            y = F.scaled_dot_product_attention(
                q * math.sqrt(hs) * mup_scale,   # pre-scale q so net scale = 1/d_head
                k, v,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=True,
            )
        else:
            att = (q @ k.transpose(-2, -1)) * mup_scale
            att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_drop(att)
            y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))


class MuMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.drop   = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.drop(self.c_proj(self.gelu(self.c_fc(x))))


class MuBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = MuCausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp  = MuMLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class MuGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([MuBlock(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        # mup output layer: MuReadout applies 1/d_model scaling (no weight tying for simplicity)
        self.lm_head = MuReadout(config.n_embd, config.vocab_size, bias=False)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos  = torch.arange(T, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

mup installed.
Device: cuda  |  AMP: torch.bfloat16
Loading data
train: 145,573,946 tokens  |  val: 1,482,782 tokens


In [17]:
# Build base shapes per config
# base/delta must have the same n_layer as the target model so that
# all parameter names match. Only n_embd varies to establish the
# width-scaling axes that µP needs.

def make_mup_model(cfg):
    """
    Create a MuGPT, set base shapes, and initialize weights.
    """
    model = MuGPT(cfg)
    # n_head=4 always divides MUP_BASE_WIDTH=64 and MUP_BASE_WIDTH*2=128
    base_cfg  = GPTConfig(n_embd=MUP_BASE_WIDTH, n_layer=cfg.n_layer, n_head=4,
                          vocab_size=cfg.vocab_size, block_size=cfg.block_size)
    delta_cfg = GPTConfig(n_embd=MUP_BASE_WIDTH * 2, n_layer=cfg.n_layer, n_head=4,
                          vocab_size=cfg.vocab_size, block_size=cfg.block_size)
    base_m  = MuGPT(base_cfg)
    delta_m = MuGPT(delta_cfg)
    base_shapes = make_base_shapes(base_m, delta_m, savefile=None)
    set_base_shapes(model, base_shapes)
    del base_m, delta_m
    model.apply(model._init_weights)
    for pn, p in model.named_parameters():
        if pn.endswith('c_proj.weight'):
            nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))
    return model

# Training utilities

def get_batch(data, batch_size, block_size, dev):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(dev), y.to(dev)


def cosine_lr(step, lr_max, lr_min, warmup_steps, total_steps):
    if step < warmup_steps:
        return lr_max * (step + 1) / warmup_steps
    if step >= total_steps:
        return lr_min
    t = (step - warmup_steps) / (total_steps - warmup_steps)
    return lr_min + 0.5 * (lr_max - lr_min) * (1.0 + math.cos(math.pi * t))


@torch.no_grad()
def eval_val_loss(model, num_batches=60):
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = get_batch(val_data, MICRO_BATCH, BLOCK_SIZE, device)
        with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device=='cuda')):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def _latest_mup_ckpt(name):
    """
    Return path and step of the latest incremental µP checkpoint, or (None, 0).
    """
    import glob
    paths = sorted(glob.glob(f'{CKPT_DIR}/mup_{name}_step*.pt'))
    if not paths:
        return None, 0
    latest = paths[-1]
    step = int(os.path.basename(latest).split('step')[1].replace('.pt', ''))
    return latest, step


def run_mup_training(config, lr, total_steps, seed=42, verbose=True, name=None):
    """
    Train a µP model. If name is given, saves incremental checkpoints and supports resume.
    """
    torch.manual_seed(seed)
    model = make_mup_model(config).to(device)
    optimizer = MuAdamW(model.parameters(), lr=lr, weight_decay=0.1, betas=(0.9, 0.95), eps=1e-8)
    scaler = torch.amp.GradScaler('cuda', enabled=(device=='cuda' and not use_bf16))
    warmup_steps = max(1, int(total_steps * WARMUP_FRAC))
    lr_min = lr * 0.1

    loss_curve = []
    running_loss = 0.0
    elapsed_offset = 0.0
    peak_mem_gb = 0.0
    start_step = 0

    # Resume from latest incremental checkpoint if name is provided
    if name:
        resume_path, start_step = _latest_mup_ckpt(name)
        if resume_path:
            print(f"\nResuming mup-{name} from step {start_step:,}  ({resume_path})")
            ckpt_data = torch.load(resume_path, map_location=device)
            model.load_state_dict(ckpt_data['model_state'])
            optimizer.load_state_dict(ckpt_data['optimizer_state'])
            scaler.load_state_dict(ckpt_data['scaler_state'])
            loss_curve = ckpt_data.get('loss_curve', [])
            elapsed_offset = ckpt_data.get('elapsed_s', 0.0)
            peak_mem_gb = ckpt_data.get('peak_mem_gb', 0.0)

    t0 = time.time()

    for step in range(start_step, total_steps):
        lr_now = cosine_lr(step, lr, lr_min, warmup_steps, total_steps)
        for g in optimizer.param_groups:
            g['lr'] = lr_now

        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0
        for _ in range(GRAD_ACCUM):
            x, y = get_batch(train_data, MICRO_BATCH, BLOCK_SIZE, device)
            with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device=='cuda')):
                _, loss = model(x, y)
                loss = loss / GRAD_ACCUM
            scaler.scale(loss).backward()
            step_loss += loss.item()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += step_loss

        if device == 'cuda':
            peak_mem_gb = max(peak_mem_gb, torch.cuda.max_memory_allocated() / 1e9)

        if (step + 1) % LOG_INTERVAL == 0:
            avg_t = running_loss / LOG_INTERVAL
            running_loss = 0.0
            tps = (step + 1 - start_step) * MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE / (time.time() - t0)
            loss_curve.append({'step': step+1, 'train_loss': avg_t})
            if verbose:
                print(f"[{step+1:>5}/{total_steps}]  train={avg_t:.4f}  lr={lr_now:.2e}  {tps/1e3:.1f}K tok/s")

        if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == total_steps:
            val_loss = eval_val_loss(model)
            if loss_curve:
                loss_curve[-1]['val_loss'] = val_loss
            if verbose:
                print(f"val_loss = {val_loss:.4f}")

        # Incremental checkpoint (scaling runs only, not LR sweep)
        if name and (step + 1) % CKPT_INTERVAL == 0 and (step + 1) < total_steps:
            inc_path = f'{CKPT_DIR}/mup_{name}_step{step+1}.pt'
            torch.save({
                'config': config.__dict__,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scaler_state': scaler.state_dict(),
                'step':  step + 1,
                'loss_curve':  loss_curve,
                'elapsed_s': elapsed_offset + time.time() - t0,
                'peak_mem_gb': peak_mem_gb,
            }, inc_path)
            if verbose:
                print(f"  ✓ Incremental checkpoint → {inc_path}")

    wall_time = elapsed_offset + time.time() - t0
    final_val = eval_val_loss(model, num_batches=200)
    return model, final_val, loss_curve, wall_time, peak_mem_gb


In [18]:

# mup LR sweep on Tiny

SWEEP_FILE = f'{RESULTS_DIR}/lr_sweep_mup.json'

if os.path.exists(SWEEP_FILE):
    with open(SWEEP_FILE) as f:
        sweep_results = json.load(f)
    done_lrs = {r['lr'] for r in sweep_results}
    print(f"Resuming mup sweep with {len(done_lrs)}/{len(LR_CANDIDATES)} done.")
else:
    sweep_results = []
    done_lrs = set()

cfg_tiny = GPTConfig(n_embd=128, n_layer=4, n_head=4,
                     vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE)
params_tiny = make_mup_model(cfg_tiny).num_params()
print(f"\n mup LR sweep on Tiny ({params_tiny:,} params, {SWEEP_STEPS} steps each):\n")

for lr in LR_CANDIDATES:
    if lr in done_lrs:
        print(f"lr={lr:.0e}  (skipped)")
        continue
    print(f"\n mup lr={lr:.0e}")
    _, final_val, loss_log, elapsed, _ = run_mup_training(cfg_tiny, lr, SWEEP_STEPS)
    result = {'lr': lr, 'final_val_loss': final_val, 'elapsed_s': round(elapsed, 1), 'loss_log': loss_log}
    sweep_results.append(result)
    with open(SWEEP_FILE, 'w') as f:
        json.dump(sweep_results, f, indent=2)
    print(f"Final val = {final_val:.4f}")

best_mup = min(sweep_results, key=lambda r: r['final_val_loss'])
print(f"\nBest mup LR: {best_mup['lr']:.2e}  (val={best_mup['final_val_loss']:.4f})")
with open(f'{RESULTS_DIR}/best_lr_mup.json', 'w') as f:
    json.dump({'lr': best_mup['lr'], 'val_loss': best_mup['final_val_loss']}, f)

# Train all mup model sizes

MUP_RESULT_FILE = f'{RESULTS_DIR}/mup_scaling_results.json'
BEST_LR_MUP = best_mup['lr']

if os.path.exists(MUP_RESULT_FILE):
    with open(MUP_RESULT_FILE) as f:
        mup_results = json.load(f)
    done = {r['name'] for r in mup_results}
    print(f"\nResuming mup training as {len(done)} model(s) already done: {done}")
else:
    mup_results = []
    done = set()

for name, base_cfg in MODEL_CONFIGS.items():
    if name in done:
        print(f"Skipping mup-{name} (done).")
        continue

    cfg = GPTConfig(
        block_size=BLOCK_SIZE, vocab_size=VOCAB_SIZE,
        n_embd=base_cfg.n_embd, n_layer=base_cfg.n_layer,
        n_head=base_cfg.n_head, dropout=0.0, bias=False,
    )

    print(f"\n{'='*60}")
    print(f"µP Training: {name}  ({make_mup_model(cfg).num_params():,} params)  lr={BEST_LR_MUP:.2e}")
    print(f"{'='*60}")

    model, final_val, loss_curve, wall_time, peak_mem = run_mup_training(
        cfg, BEST_LR_MUP, STEPS_PER_EPOCH, name=name
    )

    # Save checkpoint
    ckpt_path = f'{CKPT_DIR}/mup_{name}.pt'
    torch.save({'config': cfg.__dict__, 'model_state': model.state_dict(),
                'final_val_loss': final_val}, ckpt_path)

    result = {
        'name': name,
        'params': model.num_params(),
        'final_val_loss': final_val,
        'wall_time_s': round(wall_time, 1),
        'peak_mem_gb': round(peak_mem, 2),
        'tokens_per_sec': round(STEPS_PER_EPOCH * MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE / wall_time),
        'loss_curve': loss_curve,
        'config': {'n_embd': cfg.n_embd, 'n_layer': cfg.n_layer, 'n_head': cfg.n_head},
    }
    mup_results.append(result)
    with open(MUP_RESULT_FILE, 'w') as f:
        json.dump(mup_results, f, indent=2)
    print(f"\n mup-{name}  val_loss={final_val:.4f}  time={wall_time/60:.1f}m")

# Summary

print(f"\n{'='*70}")
print("mup Scaling Study Summary")
print(f"{'='*70}")
print(f"{'Model':>8}  {'Params':>12}  {'Val Loss':>9}  {'Time':>8}  {'Tok/s':>8}")
print("-" * 60)
for r in sorted(mup_results, key=lambda x: x['params']):
    print(f"{r['name']:>8}  {r['params']:>12,}  {r['final_val_loss']:>9.4f}  "
          f"{r['wall_time_s']/60:>7.1f}m  {r['tokens_per_sec']/1e3:>7.1f}K")


Resuming mup sweep with 7/7 done.

 mup LR sweep on Tiny (1,901,696 params, 400 steps each):

lr=1e-04  (skipped)
lr=3e-04  (skipped)
lr=6e-04  (skipped)
lr=1e-03  (skipped)
lr=3e-03  (skipped)
lr=6e-03  (skipped)
lr=1e-02  (skipped)

Best mup LR: 6.00e-03  (val=1.1457)

Resuming mup training as 5 model(s) already done: {'tiny', 'small', 'xl', 'large', 'medium'}
Skipping mup-tiny (done).
Skipping mup-small (done).
Skipping mup-medium (done).
Skipping mup-large (done).
Skipping mup-xl (done).

mup Scaling Study Summary
   Model        Params   Val Loss      Time     Tok/s
------------------------------------------------------------
    tiny     1,901,696     0.7341     23.5m    103.1K
   small     4,327,872     0.6976      4.6m    530.4K
  medium    13,964,160     0.6774      3.9m    619.4K
   large    35,924,480     0.6557      7.2m    335.2K
      xl    91,638,528     0.6944     12.5m    194.0K


In [22]:
# SP vs mup comparison + scaling law extrapolation
# creates comparison_plot.png, lr_sweep_comparison.png

import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import t as t_dist

BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'
RESULTS_DIR = f'{BASE_DIR}/results'


# Load results

with open(f'{RESULTS_DIR}/scaling_results.json') as f: sp_results  = json.load(f)
with open(f'{RESULTS_DIR}/mup_scaling_results.json') as f: mup_results = json.load(f)
with open(f'{RESULTS_DIR}/lr_sweep_tiny.json') as f: lr_sweep_sp  = json.load(f)
with open(f'{RESULTS_DIR}/lr_sweep_mup.json') as f: lr_sweep_mup = json.load(f)

sp_results.sort(key=lambda r: r['params'])
mup_results.sort(key=lambda r: r['params'])

sp_params = np.array([r['params'] for r in sp_results],  dtype=float)
sp_losses = np.array([r['final_val_loss'] for r in sp_results],  dtype=float)
mup_params = np.array([r['params'] for r in mup_results], dtype=float)
mup_losses = np.array([r['final_val_loss'] for r in mup_results], dtype=float)

In [23]:
# Power-law fitting
# With only 5 points and 3 free parameters, the full L = a·N^{-alpha}+c fit
# is degenerate when losses are close together (covariance diverges).
# I use a log-log linear regression (2 params, always stable) as the
# primary estimate, then attempt a 3-param fit with c fixed as a check.

import math

def power_law(N, a, alpha, c):
    return a * N**(-alpha) + c

def power_law_2(N, a, alpha):
    return a * N**(-alpha)

def fit_power_law(params, losses, label):
    ss_tot = np.sum((losses - losses.mean())**2)

    # Method 1: log-log linear regression (most stable)
    log_N = np.log10(params)
    log_L = np.log10(losses)
    slope, intercept = np.polyfit(log_N, log_L, 1)
    alpha_ll = -slope
    a_ll = 10 ** intercept
    y_pred_ll = power_law_2(params, a_ll, alpha_ll)
    r2_ll = 1 - np.sum((losses - y_pred_ll)**2) / ss_tot

    print(f"\n{label} power law (log-log):  L = {a_ll:.3f} · N^({alpha_ll:.4f})")
    print(f"alpha = {alpha_ll:.4f} R2= {r2_ll:.4f}")

    # Method 2: 3-param fit with c fixed at 95% of min loss
    c_fixed = losses.min() * 0.95
    try:
        popt2, pcov2 = curve_fit(
            power_law_2, params, losses - c_fixed,
            p0=[a_ll, alpha_ll],
            bounds=([0, 0], [1e8, 3.0]),
            maxfev=10000,
        )
        a_f, alpha_f = popt2
        perr2 = np.sqrt(np.diag(pcov2))
        y_pred_f = power_law(params, a_f, alpha_f, c_fixed)
        r2_f = 1 - np.sum((losses - y_pred_f)**2) / ss_tot
        print(f"{label} power law (c={c_fixed:.3f} fixed):  L = {a_f:.3f} · N^({alpha_f:.4f}) + {c_fixed:.4f}")
        print(f"alpha= {alpha_f:.4f} +- {perr2[1]:.4f} R2 = {r2_f:.4f}")
        popt_full = (a_f, alpha_f, c_fixed)
        perr_full = [0, perr2[1], 0]
    except Exception as e:
        print(f"3-param fit failed ({e}) using log-log only.")
        popt_full = (a_ll, alpha_ll, 0.0)
        perr_full = [0, 0, 0]

    # Return log-log alpha as primary (more reliable), full popt for plotting
    return alpha_ll, r2_ll, popt_full, perr_full

In [24]:
sp_alpha,  sp_r2,  sp_popt,  sp_perr  = fit_power_law(sp_params,  sp_losses,  'SP')
mup_alpha, mup_r2, mup_popt, mup_perr = fit_power_law(mup_params, mup_losses, 'µP')
sp_ok = True; mup_ok = True

# Comparison scaling plot

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(sp_params,  sp_losses,  s=80, color='steelblue', zorder=5, label='SP (measured)')
ax.scatter(mup_params, mup_losses, s=80, color='darkorange', marker='D', zorder=5, label='mup (measured)')

N_range = np.logspace(np.log10(min(sp_params.min(), mup_params.min())*0.5),
                      np.log10(max(sp_params.max(), mup_params.max())*4), 400)

ax.plot(N_range, power_law(N_range, *sp_popt), 'b--', lw=1.5,
        label=fr'SP fit: alpha={sp_alpha:.3f}  (R2={sp_r2:.2f})')
ax.plot(N_range, power_law(N_range, *mup_popt), 'r--', lw=1.5,
        label=fr'µP fit: alpha={mup_alpha:.3f}  (R2={mup_r2:.2f})')


SP power law (log-log):  L = 0.030 · N^(-0.2139)
alpha = -0.2139 R2= 0.7723
3-param fit failed (Initial guess is outside of provided bounds) using log-log only.

µP power law (log-log):  L = 0.923 · N^(0.0176)
alpha = 0.0176 R2= 0.4533
µP power law (c=0.623 fixed):  L = 2.492 · N^(0.2224) + 0.6229
alpha= 0.2224 +- 0.1141 R2 = 0.5458


In [25]:
# Annotate model names
for r in sp_results:
    ax.annotate(r['name'], (r['params'], r['final_val_loss']),textcoords='offset points', xytext=(6, 4), fontsize=8, color='steelblue')

ax.set_xscale('log')
ax.set_xlabel('Number of Parameters (log scale)')
ax.set_ylabel('Validation Loss (1 epoch)')
ax.set_title('Scaling Curves: Standard Parameterization vs. mup')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/comparison_scaling_plot.png', dpi=150)
plt.show()
print(f"Saved comparison_scaling_plot.png")

# LR sweep comparison

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, title, color in [
    (axes[0], lr_sweep_sp,  'SP LR Sweep (Tiny)',  'steelblue'),
    (axes[1], lr_sweep_mup, 'mup LR Sweep (Tiny)',  'darkorange'),
]:
    lrs = [r['lr'] for r in sorted(data, key=lambda x: x['lr'])]
    vls = [r['final_val_loss'] for r in sorted(data, key=lambda x: x['lr'])]
    ax.semilogx(lrs, vls, 'o-', color=color, ms=7, lw=2)
    best_idx = int(np.argmin(vls))
    ax.scatter([lrs[best_idx]], [vls[best_idx]], s=150, marker='*', color='red', zorder=6, label=f'Best: {lrs[best_idx]:.0e}')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Validation Loss')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/lr_sweep_comparison.png', dpi=150)
plt.show()
print(f"Saved to lr_sweep_comparison.png")

Saved comparison_scaling_plot.png
Saved to lr_sweep_comparison.png


In [27]:

# Scaling law extrapolation

# Use the better-fitting curve (higher R2) for extrapolation
if mup_r2 >= sp_r2:
    best_popt, best_alpha, best_r2, best_label = mup_popt, mup_alpha, mup_r2, 'mup'
else:
    best_popt, best_alpha, best_r2, best_label = sp_popt,  sp_alpha,  sp_r2,  'SP'
best_perr = [0, 0, 0]  # perr not meaningful for log-log

if best_popt is not None:
    xl_params = max(sp_params.max(), mup_params.max())
    target_params = xl_params * 10.0   # 10× XL

    predicted_loss = power_law(target_params, *best_popt)

    # Uncertainty: propagate error in α (dominant term)
    a, alpha, c = best_popt
    dL_dalpha = a * target_params**(-alpha) * math.log(target_params)
    loss_uncertainty = dL_dalpha * best_perr[1]

    print(f"\n{'='*60}")
    print(f"Scaling Law Extrapolation (using {best_label} fit)")
    print(f"{'='*60}")
    print(f"Largest trained model: {xl_params/1e6:.1f}M params with val_loss = {power_law(xl_params, *best_popt):.4f}")
    print(f"10x larger prediction:  {target_params/1e6:.1f}M params with val_loss ≈ {predicted_loss:.4f} +- {loss_uncertainty:.4f}")
    # confidence interval a range that, if you repeated the experiment many times, would contain the true parameter value 95% of the time.
    # print(f"\n95% CI: [{predicted_loss - 2*loss_uncertainty:.4f}, {predicted_loss + 2*loss_uncertainty:.4f}]")
    print()
    print("Caveats on extrapolation reliability:")
    # Power laws typically hold over ~2–3 orders of magnitude of N"
    print(f"The fit spans {np.log10(sp_params.max()/sp_params.min()):.1f} orders; extrapolating 1 order further")
    # SVG domain may have a lower loss floor than NLP due to finite vocabulary
    # Data volume (100M tokens) may become the bottleneck before model size
    # Architecture changes (e.g., depth vs. width ratios) may shift the curve

    extrapolation = {
        'source_parameterization': best_label,
        'xl_params': float(xl_params),
        'target_params': float(target_params),
        'predicted_loss': float(predicted_loss),
        'uncertainty_1sigma': float(loss_uncertainty),
        'fit_params': {'a': float(a), 'alpha': float(alpha), 'c': float(c)},
    }
    with open(f'{RESULTS_DIR}/extrapolation.json', 'w') as f:
        json.dump(extrapolation, f, indent=2)

# Analysis: did mup help?

print(f"\n{'='*60}")
print("SP vs. mup comparison")
print(f"{'='*60}")
print(f"{'Model':>8}  {'SP Loss':>9}  {'mup Loss':>9}  {'diff (µP-SP)':>10}")
print("-" * 45)
sp_by_name = {r['name']: r['final_val_loss'] for r in sp_results}
mup_by_name = {r['name']: r['final_val_loss'] for r in mup_results}
for name in ['tiny', 'small', 'medium', 'large', 'xl']:
    if name in sp_by_name and name in mup_by_name:
        sp_l = sp_by_name[name]
        mu_l = mup_by_name[name]
        delta = mu_l - sp_l
        sign = ' better' if delta < 0 else (' worse' if delta > 0 else '=')
        print(f"{name:>8}  {sp_l:>9.4f}  {mu_l:>9.4f}  {delta:>+10.4f}  {sign}")

print(f"\nSP  scaling exponent alpha = {sp_alpha:.4f}  R2 = {sp_r2:.4f}")
print(f"mup scaling exponent alpha = {mup_alpha:.4f}   R2 = {mup_r2:.4f}")
if mup_alpha > sp_alpha:
    print("mup shows steeper scaling (larger alpha = loss drops faster with N).")
else:
    print("SP and mup show similar scaling and mup advantage may be in LR transfer.")


Scaling Law Extrapolation (using SP fit)
Largest trained model: 91.6M params with val_loss = 1.5317
10x larger prediction:  916.4M params with val_loss ≈ 2.5065 +- 0.0000

Caveats on extrapolation reliability:
The fit spans 2.0 orders; extrapolating 1 order further

SP vs. mup comparison
   Model    SP Loss   mup Loss  diff (µP-SP)
---------------------------------------------
    tiny     0.7226     0.7341     +0.0114   worse
   small     0.6502     0.6976     +0.0474   worse
  medium     0.6322     0.6774     +0.0452   worse
   large     1.3587     0.6557     -0.7030   better
      xl     1.8051     0.6944     -1.1107   better

SP  scaling exponent alpha = -0.2139  R2 = 0.7723
mup scaling exponent alpha = 0.0176   R2 = 0.4533
mup shows steeper scaling (larger alpha = loss drops faster with N).
